In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

import joblib

import warnings
warnings.filterwarnings("ignore")

In [ ]:
df = pd.read_csv(r"..\data\AmesHousing.csv")

df.head()

In [ ]:
print("Shape:", df.shape)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum().sum()

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing

In [ ]:
print("Duplicate Rows:", df.duplicated().sum()) #no duplicates

In [ ]:
#features not available, so instead fill with None, for categorical columns
none_cols = [
    'Pool QC','Misc Feature','Alley','Fence',
    'Fireplace Qu','Garage Type','Garage Finish',
    'Garage Qual','Garage Cond',
    'Bsmt Qual','Bsmt Cond','Bsmt Exposure',
    'BsmtFin Type 1','BsmtFin Type 2',
    'Mas Vnr Type'
]

for col in none_cols:
    if col in df.columns:
        df[col] = df[col].fillna("None")

In [ ]:
#for numerical column
df["Garage Yr Blt"] = df["Garage Yr Blt"].fillna(0)
df["Mas Vnr Area"] = df["Mas Vnr Area"].fillna(0)

In [ ]:
num_cols = df.select_dtypes(include=['int64','float64']).columns

for col in num_cols:
    df[col]=df[col].fillna(df[col].median())

In [ ]:
cat_cols = df.select_dtypes(include=['object']).columns

for col in cat_cols:
    df[col]=df[col].fillna(df[col].mode()[0])

In [ ]:
df.isnull().sum().sum()

In [ ]:
#Univariet graph
plt.figure(figsize=(8,5))

sns.histplot(df["SalePrice"], bins=30, kde=True) #bins divide the SalePrice into 30 equal intervals
#kde adds the smooth density curve

plt.title("Distribution of SalePrice")

plt.show()

In [ ]:
#correlation heatmap for all numerical columns
#to know which features are related to house price
#for model training
plt.figure(figsize=(14,10))

corr = df.corr(numeric_only=True)
#ranges from -1 to +1
sns.heatmap(corr, cmap="coolwarm")

plt.title("Correlation Heatmap")

plt.show()

In [ ]:
corr = df.corr(numeric_only=True)

top_corr = corr["SalePrice"].sort_values(ascending=False)

print(top_corr.head(15))

In [ ]:
#how strongly feature is correlated with SalePrice
top_features = top_corr[1:11]

plt.figure(figsize=(10,5))
sns.barplot(
    x=top_features.values,
    y=top_features.index
)
plt.title("Top Features Correlated with SalePrice")
plt.show()

In [ ]:
cols = [
    "SalePrice",
    "Overall Qual",
    "Gr Liv Area",
    "Garage Cars",
    "Total Bsmt SF"
]

sns.pairplot(df[cols])

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(x=df["SalePrice"])

plt.title("SalePrice Outliers")

plt.show()

In [ ]:
features = [
    "Overall Qual",
    "Gr Liv Area",
    "Garage Cars",
    "Garage Area",
    "Year Built",
    "Full Bath",
    "Lot Area",
    "Total Bsmt SF"
]

X = df[features]
y = df["SalePrice"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)

In [ ]:
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("Linear Regression Performance")
print("-" * 35)
print(f"R² Score : {r2:.4f}")
print(f"MAE       : {mae:.2f}")
print(f"RMSE      : {rmse:.2f}")

In [ ]:
results = pd.DataFrame({
    'Actual Price': y_test,
    'Predicted Price': y_pred
})

results.head(10)

In [ ]:
plt.figure(figsize=(7,6))

plt.scatter(y_test, y_pred, alpha=0.7)

plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    color="red"
)

plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Actual vs Predicted House Prices")

plt.show()

In [ ]:
joblib.dump(
    lr,
    "../model/house_price_model.pkl"
)

print("Model saved successfully!")